<a href="https://colab.research.google.com/github/lubbad-aljazeera/paid-activity-data-pipeline/blob/main/X_By_Country_Data_Pipeline_Extend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.cloud import storage, bigquery
from google.colab import auth
import re
import io
import numpy as np
import csv
from datetime import datetime # Import datetime for backup table naming

auth.authenticate_user()
print("✅ Google Cloud authenticated successfully.")

bucket_name = "aj360_data"
dataset_id = "aj360"
table_id = "x_ads_country"
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
table_ref = dataset_ref.table(table_id)
clean_file = "x_ads_clean.csv"

prefix = "X_ADS/Location/x_ads_country_5-7Mar2026.csv"
print(f"Listing CSV files in bucket: {bucket_name} with prefix: {prefix}")
storage_client = storage.Client(project=project_id)
bucket = storage_client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=prefix))
csv_files = [blob.name for blob in blobs if blob.name.endswith('.csv')]

if not csv_files:
    print("❌ No CSV files found.")
else:
    print(f"✅ Found {len(csv_files)} CSV files to process.")

    bq_client = bigquery.Client(project=project_id)
    try:
        bq_client.get_dataset(dataset_ref)
        print(f"✅ Dataset {dataset_id} exists.")
    except:
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        dataset = bq_client.create_dataset(dataset)
        print(f"✅ Created dataset {dataset_id}.")

    print("Loading lookup table from BigQuery...")
    try:
        lookup_df = bq_client.query("""
            SELECT term, show_title_ar, show_title_en
            FROM `ajgc-dig-jwc-prod-jawacloud-01.aj360.paid_ads_shows_lookup`
        """).to_dataframe()
        term_lookup = dict(zip(lookup_df['term'], lookup_df['show_title_en']))
        print(f"✅ Loaded lookup table with {len(term_lookup)} terms.")
    except Exception as e:
        print(f"❌ Failed to load lookup table: {e}")
        csv_files = []

    # Fetch existing table schema for comparison and adjustment
    existing_table = bq_client.get_table(table_ref)
    existing_schema_fields = {field.name: field for field in existing_table.schema}

    unified_schema = {}
    schema = None

    print("\n--- Schema Inference ---")
    for file in csv_files[:5]:
        try:
            print(f"Inferring schema from: {file}")
            blob = bucket.blob(file)
            blob.download_to_filename("/tmp/schema.csv")
            df_for_schema_inference = pd.read_csv("/tmp/schema.csv", encoding='utf-8', on_bad_lines='skip')

            if df_for_schema_inference.empty:
                continue

            df_for_schema_inference.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col).lower() for col in df_for_schema_inference.columns]

            # Prepare df_for_schema_inference to more accurately reflect BigQuery types
            # by attempting to convert numeric-like and datetime-like columns
            for col in df_for_schema_inference.columns:
                if df_for_schema_inference[col].dtype == 'object':
                    # Try to clean and convert to numeric first
                    # Remove commas and strip whitespace, then convert to numeric
                    cleaned_series = df_for_schema_inference[col].astype(str).str.replace(',', '', regex=False).str.strip()
                    numeric_series = pd.to_numeric(cleaned_series, errors='coerce')
                    if not numeric_series.isna().all():
                        df_for_schema_inference[col] = numeric_series
                        continue # If successfully converted to numeric, move to next column

                    # If not numeric, try to convert to datetime
                    datetime_series = pd.to_datetime(df_for_schema_inference[col], errors='coerce')
                    if not datetime_series.isna().all():
                        df_for_schema_inference[col] = datetime_series
                        continue # If successfully converted to datetime, move to next column

            # Add 'show_title' to the dataframe for schema inference if it's expected in the final schema
            if 'ad_group_name' in df_for_schema_inference.columns and 'show_title' not in df_for_schema_inference.columns:
                df_for_schema_inference['show_title'] = None

            # Now infer schema based on the potentially converted dtypes of df_for_schema_inference
            for col, dtype in df_for_schema_inference.dtypes.items():
                if col not in unified_schema:
                    if 'float' in str(dtype): # Check for float after conversion attempt
                        bq_type = 'FLOAT'
                    elif 'int' in str(dtype): # Check for int after conversion attempt
                        bq_type = 'INTEGER'
                    elif 'bool' in str(dtype): # Check for boolean
                        bq_type = 'BOOLEAN'
                    elif 'datetime' in str(dtype): # Check for datetime
                        if col == 'time_period':
                            bq_type = 'STRING'
                        else:
                            bq_type = 'TIMESTAMP'
                    else:
                        bq_type = 'STRING'
                    unified_schema[col] = bq_type
        except Exception as e:
            print(f"❌ Failed schema inference on {file}: {e}")

    if not unified_schema:
        print("❌ Schema inference failed. Exiting.")
        exit()

    # Adjust unified_schema to match existing BigQuery table schema
    final_schema_fields = []
    for field_name, existing_field in existing_schema_fields.items():
        if field_name in unified_schema: # Use inferred type if available
            final_schema_fields.append(bigquery.SchemaField(existing_field.name, unified_schema[field_name], existing_field.mode))
        else: # Otherwise, use existing BigQuery field definition
            final_schema_fields.append(existing_field)

    # Add fields from inferred schema that are not in existing table schema, if applicable.
    # For this problem, 'media_views' was the problematic field, so we prioritize the existing schema.
    # We need to explicitly exclude 'media_views' if it's not in the existing schema.
    # And ensure 'objective_1' from existing schema is present.

    # Filter out 'media_views' from the inferred schema if it's not in the existing table
    if 'media_views' in unified_schema and 'media_views' not in existing_schema_fields:
        del unified_schema['media_views']

    # Construct the final schema based on the existing table, then add new fields from inferred schema that are not in existing table
    # We will prioritize the existing schema to avoid 'Cannot add fields' error.
    # Any fields from the inferred schema that are *not* in the existing table schema will be ignored for this append operation.
    schema = []
    for field_name, existing_field in existing_schema_fields.items():
        # Use the existing field definition directly to maintain compatibility
        schema.append(existing_field)

    # Optionally, you might want to add *new* fields from unified_schema if they don't exist in existing_schema_fields
    # and you are sure they should be added (which usually requires a schema update in BQ, not just WRITE_APPEND)
    # For this fix, we will stick to the existing schema to avoid the error.

    # Re-map unified_schema to match existing BigQuery schema order and types exactly
    schema = [existing_schema_fields[field.name] for field in existing_table.schema]

    if csv_files and schema: # Only proceed with data processing if files and schema are available
        # Generates a string like '2026_02_26' (Recommended for better sorting)
        current_date_string = datetime.now().strftime('%Y_%m_%d')
        backup_table_name = f"{table_id}_backup_{current_date_string}"
        backup_table_ref = dataset_ref.table(backup_table_name)
        backup_query = f"""
        CREATE TABLE IF NOT EXISTS `{project_id}.{dataset_id}.{backup_table_name}` AS
        SELECT * FROM `{project_id}.{dataset_id}.{table_id}`;
        """
        print(f"\n--- Creating BigQuery Backup Table: {backup_table_name} ---")
        try:
            bq_client.query(backup_query).result()
            print(f"✅ Backup table '{backup_table_name}' created successfully (if it didn't exist) or already exists.")
        except Exception as e:
            print(f"❌ Failed to create backup table '{backup_table_name}': {e}")
            # Decide if processing should halt on backup failure; for now, print error and continue
        # --- End Backup Mechanism ---

    print("✅ Final Schema (aligned with existing BigQuery table):")
    for s in schema:
        print(f"  {s.name}: {s.field_type}")

    print("\n--- Starting Data Processing and Loading Pass ---")
    for i, source_file in enumerate(csv_files):
        print(f"\nProcessing: {source_file}")
        try:
            blob = bucket.blob(source_file)
            blob.download_to_filename("/tmp/source.csv")
            df = pd.read_csv("/tmp/source.csv", encoding='utf-8', on_bad_lines='skip')

            if df.empty:
                print("⚠️ Empty file. Skipping.")
                continue

            df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', col).lower() for col in df.columns]

            if 'ad_group_name' in df.columns:
                df['ad_group_name'] = df['ad_group_name'].astype(str)
                split_result = df['ad_group_name'].str.split('_', expand=True)
                df['part2'] = split_result[1].fillna('')
                df['part3'] = split_result[2].fillna('') if split_result.shape[1] > 2 else ''

            if 'campaign_name' in df.columns:
                df['campaign_name'] = df['campaign_name'].astype(str)
                split_result = df['campaign_name'].str.split('_', expand=True)
                df['cpart2'] = split_result[1].fillna('')
                df['cpart3'] = split_result[2].fillna('') if split_result.shape[1] > 2 else ''

                def extract_show_title(row):
                    for part in [row.get('part2'), row.get('part3'), row.get('cpart2'), row.get('cpart3')]:
                        if part in term_lookup:
                            return term_lookup[part]
                    return None
                df['show_title'] = df.apply(extract_show_title, axis=1)
                df.drop(columns=['part2', 'part3','cpart2', 'cpart3'], inplace=True, errors='ignore')
            else:
                df['show_title'] = None

            # Ensure df has all columns from the BigQuery schema, adding None for missing ones
            # And ensure order matches BigQuery schema
            all_bq_schema_cols = [field.name for field in schema]
            for col in all_bq_schema_cols:
                if col not in df.columns:
                    df[col] = None
            df = df[all_bq_schema_cols]

            for field in schema:
                col = field.name
                typ = field.field_type
                if typ == 'INTEGER':
                    # Clean numeric columns before conversion to avoid errors during astype
                    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
                elif typ == 'FLOAT':
                    # Clean numeric columns before conversion to avoid errors during astype
                    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                elif typ == 'BOOLEAN':
                    df[col] = df[col].astype('boolean')
                elif typ == 'TIMESTAMP':
                    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
                else:
                    df[col] = df[col].astype(str).replace('nan', None)

            for col in df.select_dtypes(include='object').columns:
                df[col] = df[col].astype(str).str.replace('"', '').str.replace("'", '').str.replace('\n', '').str.replace('\r', '')
                df[col] = df[col].replace('nan', None)

            df.to_csv("/tmp/clean.csv", index=False, sep=',', quoting=csv.QUOTE_MINIMAL, encoding='utf-8', header=True)
            blob_clean = bucket.blob(clean_file)
            blob_clean.upload_from_filename("/tmp/clean.csv")
            print("✅ Cleaned file uploaded to GCS.")

            write_disposition = bigquery.WriteDisposition.WRITE_APPEND
            load_config = bigquery.LoadJobConfig(
                source_format=bigquery.SourceFormat.CSV,
                skip_leading_rows=1,
                schema=schema, # Use the aligned schema
                write_disposition=write_disposition,
                encoding='UTF-8',
                quote_character='"',
                field_delimiter=',',
                allow_quoted_newlines=True,
                allow_jagged_rows=True,
                max_bad_records=1000
            )

            destination_uri = f"gs://{bucket_name}/{clean_file}"
            load_job = bq_client.load_table_from_uri(destination_uri, table_ref, job_config=load_config)
            load_job.result()
            print(f"✅ Loaded {source_file} to BigQuery.")

        except Exception as e:
            print(f"❌ Error: {e}")

✅ Google Cloud authenticated successfully.
Listing CSV files in bucket: aj360_data with prefix: X_ADS/Location/x_ads_country_5-7Mar2026.csv
✅ Found 1 CSV files to process.
✅ Dataset aj360 exists.
Loading lookup table from BigQuery...
✅ Loaded lookup table with 699 terms.

--- Schema Inference ---
Inferring schema from: X_ADS/Location/x_ads_country_5-7Mar2026.csv


/tmp/ipykernel_521/3560940980.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_series = pd.to_datetime(df_for_schema_inference[col], errors='coerce')
/tmp/ipykernel_521/3560940980.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_series = pd.to_datetime(df_for_schema_inference[col], errors='coerce')
/tmp/ipykernel_521/3560940980.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_series = pd.to_datetime(df_for_schema_inference[col], errors='coerce')
/tmp/ipykernel_521/3560940980.py:88: UserWarning: Could not infer format, so each element w


--- Creating BigQuery Backup Table: x_ads_country_backup_2026_03_08 ---
✅ Backup table 'x_ads_country_backup_2026_03_08' created successfully (if it didn't exist) or already exists.
✅ Final Schema (aligned with existing BigQuery table):
  time_period: STRING
  placement: STRING
  location: STRING
  ad_group_name: STRING
  objective: STRING
  ad_group_status: STRING
  impressions: FLOAT
  spend: FLOAT
  campaign_name: STRING
  tweet_engagements: FLOAT
  video_views: FLOAT
  total_audience_reach: STRING
  link_click_rate: STRING
  clicks: FLOAT
  objective_1: STRING
  link_clicks: FLOAT
  show_title: STRING

--- Starting Data Processing and Loading Pass ---

Processing: X_ADS/Location/x_ads_country_5-7Mar2026.csv
✅ Cleaned file uploaded to GCS.
✅ Loaded X_ADS/Location/x_ads_country_5-7Mar2026.csv to BigQuery.


In [ ]:
print("\n--- Existing BigQuery Table Schema ---")
existing_table = bq_client.get_table(table_ref)
existing_schema = existing_table.schema
for field in existing_schema:
    print(f"  {field.name}: {field.field_type}")


--- Existing BigQuery Table Schema ---
  time_period: STRING
  placement: STRING
  location: STRING
  ad_group_name: STRING
  objective: STRING
  ad_group_status: STRING
  impressions: FLOAT
  spend: FLOAT
  campaign_name: STRING
  tweet_engagements: FLOAT
  video_views: FLOAT
  total_audience_reach: STRING
  link_click_rate: STRING
  clicks: FLOAT
  objective_1: STRING
  link_clicks: FLOAT
  show_title: STRING


In [ ]:

print(f"The variable 'extract_show_title_from_hashtag' is of type: {type(extract_show_title)}")
print(f"Its value is: {extract_show_title}")

print("\nLooping through the characters of 'extract_show_title':")
for char in df['show_title']:
    print(char)

The variable 'extract_show_title_from_hashtag' is of type: <class 'function'>
Its value is: <function extract_show_title at 0x7c32d176af20>

Looping through the characters of 'extract_show_title':
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoName
NoNam